In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
%matplotlib inline

df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)
print(f"Shape: {df.shape}")

Shape: (150000, 11)


In [3]:
print(f"Rows with age = 0: {(df['age'] == 0).sum()}")
df[df['age'] == 0]

Rows with age = 0: 1


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
65696,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


In [4]:
for col in ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']:
    print(f"{col}: max = {df[col].max()}, value_counts of top values:")
    print(df[col].value_counts().head(5))
    print()

NumberOfTime30-59DaysPastDueNotWorse: max = 98, value_counts of top values:
NumberOfTime30-59DaysPastDueNotWorse
0    126018
1     16033
2      4598
3      1754
4       747
Name: count, dtype: int64

NumberOfTime60-89DaysPastDueNotWorse: max = 98, value_counts of top values:
NumberOfTime60-89DaysPastDueNotWorse
0     142396
1       5731
2       1118
3        318
98       264
Name: count, dtype: int64

NumberOfTimes90DaysLate: max = 98, value_counts of top values:
NumberOfTimes90DaysLate
0    141662
1      5243
2      1555
3       667
4       291
Name: count, dtype: int64



In [5]:
print(f"Shape before: {df.shape}")
df = df[df['age'] > 0]
print(f"Shape after dropping age=0: {df.shape}")

Shape before: (150000, 11)
Shape after dropping age=0: (149999, 11)


In [6]:
late_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 
             'NumberOfTime60-89DaysPastDueNotWorse', 
             'NumberOfTimes90DaysLate']

for col in late_cols:
    df[col] = df[col].clip(upper=10)
    print(f"{col}: new max = {df[col].max()}")

NumberOfTime30-59DaysPastDueNotWorse: new max = 10
NumberOfTime60-89DaysPastDueNotWorse: new max = 10
NumberOfTimes90DaysLate: new max = 10


In [7]:
cap_features = ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome']

for col in cap_features:
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper)
    print(f"{col}: capped at {upper:.2f}")

RevolvingUtilizationOfUnsecuredLines: capped at 1.09
DebtRatio: capped at 4979.08
MonthlyIncome: capped at 25000.00


In [8]:
from sklearn.impute import SimpleImputer

# We'll do proper pipeline imputation later in preprocess.py
# For notebook exploration, fill with median for now
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

print("Missing values after imputation:")
print(df.isnull().sum())

Missing values after imputation:
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64


In [9]:
# Combine the 3 correlated late payment columns into one
df['total_late_payments'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] + 
                              df['NumberOfTime60-89DaysPastDueNotWorse'] + 
                              df['NumberOfTimes90DaysLate'])

# Debt to income ratio (handle division by zero)
df['debt_to_income'] = df['DebtRatio'] / (df['MonthlyIncome'] + 1)

# High utilization flag (above 75% is financially stressed)
df['utilization_flag'] = (df['RevolvingUtilizationOfUnsecuredLines'] > 0.75).astype(int)

print("New features added:")
print(df[['total_late_payments', 'debt_to_income', 'utilization_flag']].describe())

New features added:
       total_late_payments  debt_to_income  utilization_flag
count        149999.000000   149999.000000     149999.000000
mean              0.453783       16.654859          0.183488
std               1.664806      210.488015          0.387067
min               0.000000        0.000000          0.000000
25%               0.000000        0.000025          0.000000
50%               0.000000        0.000065          0.000000
75%               0.000000        0.000289          0.000000
max              30.000000     4979.080000          1.000000


## Feature Engineering Summary

**Cleaning steps:**
- Dropped 1 row where age = 0 (data error) → 149,999 rows remaining
- Capped "days past due" columns at 10 (sentinel values of 96/98 removed)
- Capped RevolvingUtilization at 1.09 (99th percentile)
- Capped DebtRatio at 4979.08 (99th percentile)
- Capped MonthlyIncome at $25,000 (99th percentile)
- Imputed MonthlyIncome and NumberOfDependents with median → 0 missing values

**New features created:**
- `total_late_payments`: sum of all 3 late payment columns (replaces multicollinear trio)
- `debt_to_income`: DebtRatio / (MonthlyIncome + 1)
- `utilization_flag`: 1 if RevolvingUtilization > 75% (18.3% of borrowers flagged)

In [10]:
from sklearn.model_selection import train_test_split

TARGET = 'SeriousDlqin2yrs'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.3f}")
print(f"Test default rate:  {y_test.mean():.3f}")

Train size: (119999, 13), Test size: (30000, 13)
Train default rate: 0.067
Test default rate:  0.067


In [11]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {y_train_res.value_counts().to_dict()}")
print(f"Resampled train size: {X_train_res.shape}")

Before SMOTE: {0: 111978, 1: 8021}
After SMOTE:  {0: 111978, 1: 111978}
Resampled train size: (223956, 13)


## Train-Test Split & SMOTE

**Split:** 80/20 stratified split → 119,999 train / 30,000 test
- Both sets maintain the original 6.7% default rate (stratify=y confirmed this)

**SMOTE (applied to training set only):**
- Before: 111,978 non-defaulters vs 8,021 defaulters (14:1 imbalance)
- After: 111,978 vs 111,978 (perfectly balanced)
- Resampled training size: 223,956 rows

**Critical rule followed:** SMOTE applied AFTER splitting — test set is never touched.